# Microsoft Agent Framework による Agent 開発 (Python)

Microsoft Agent Framework SDK (Python) を使って、Microsoft Foundry のモデルに接続し、シンプルなエージェント、関数ツール、複数ターン会話、マルチエージェントワークフローを実装する方法を学びます。
- Microsoft Foundry の Project endpoint を使います
- 各セクションは個別に実行できます

In [ ]:
%pip install -U agent-framework azure-identity python-dotenv pydantic

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env を使う場合は忘れずに実行

FOUNDRY_PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
FOUNDRY_MODEL = os.getenv("FOUNDRY_MODEL", "gpt-4.1")

print("FOUNDRY_PROJECT_ENDPOINT:", "SET" if FOUNDRY_PROJECT_ENDPOINT else "NOT SET")
print("FOUNDRY_MODEL:", FOUNDRY_MODEL)

## 共通セットアップ

以下のセルで SDK を読み込み、Foundry クライアントを作成する関数を用意します。

In [ ]:
from typing import Annotated, cast

from agent_framework import Agent, Message, tool
from agent_framework.foundry import FoundryChatClient
from agent_framework.orchestrations import SequentialBuilder
from azure.identity import AzureCliCredential
from pydantic import Field

def get_foundry_client():
    if not FOUNDRY_PROJECT_ENDPOINT:
        raise ValueError("FOUNDRY_PROJECT_ENDPOINT が未設定です。.env を使う場合は load_dotenv() を実行してください。")
    return FoundryChatClient(
        project_endpoint=FOUNDRY_PROJECT_ENDPOINT,
        model=FOUNDRY_MODEL,
        credential=AzureCliCredential(),
    )

## 1. シンプルなエージェント

In [ ]:
client = get_foundry_client()

agent = Agent(
    client=client,
    name="HelloAgent",
    instructions="あなたは親切な AI アシスタントです。日本語で簡潔に回答してください。",
)

result = await agent.run("Microsoft Foundry とは何ですか？ひとことで教えてください。")
print(result)

## 2. 関数ツールの追加

In [ ]:
@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="天気を知りたい場所")],
) -> str:
    sample = {
        "Tokyo": "晴れ、最高気温 24°C",
        "Osaka": "雨、最高気温 22°C",
        "Sapporo": "くもり、最高気温 18°C",
    }
    return f"{location} の天気は {sample.get(location, '晴れ、最高気温 25°C')} です。"

tool_agent = Agent(
    client=get_foundry_client(),
    name="WeatherAgent",
    instructions="あなたは天気案内エージェントです。質問に答えるときは get_weather ツールを使ってください。",
    tools=[get_weather],
)

result = await tool_agent.run("東京の天気を教えてください。")
print(result)

## 3. 複数ターンの会話

In [ ]:
conversation_agent = Agent(
    client=get_foundry_client(),
    name="ConversationAgent",
    instructions="あなたは会話の文脈を理解する AI アシスタントです。日本語で簡潔に回答してください。",
)

session = conversation_agent.create_session()

result1 = await conversation_agent.run("私の名前は Ayako で、趣味は登山です。", session=session)
print(result1)
print()
result2 = await conversation_agent.run("私について覚えていることを教えてください。", session=session)
print(result2)

## 4. マルチエージェントワークフロー

In [ ]:
writer = Agent(
    client=get_foundry_client(),
    name="writer",
    instructions="ユーザーの依頼に対して、簡潔な説明文を日本語で作成してください。",
)

reviewer = Agent(
    client=get_foundry_client(),
    name="reviewer",
    instructions="直前の回答をレビューし、より分かりやすい改善案を日本語で返してください。",
)

workflow = SequentialBuilder(participants=[writer, reviewer]).build()

outputs: list[list[Message]] = []
async for event in workflow.run(
    "Microsoft Foundry の特徴を 3 つ挙げてください。",
    stream=True,
):
    if event.type == "output":
        outputs.append(cast(list[Message], event.data))

if outputs:
    print("===== Final Conversation =====")
    for msg in outputs[-1]:
        speaker = msg.author_name or msg.role
        if msg.text:
            print(f"[{speaker}] {msg.text}")